In [1]:
example_text = "Prior to the advent of Transformer models, word embedding served as a state-of-the-art technique for representing semantic relationships between tokens. The technique was first introduced in 2013, and it spawned a host of different variants that completely flooded the field of NLP until about 2018. In part, word embedding’s popularity stems from the relatively simple intuition behind it, which is known as the distributional hypothesis: “you shall know a word by the company it keeps!” (J.R. Firth). Words that appear in similar contexts, in other words, have similar meanings, and what word embeddings do is represent that context-specific information through a set of features. As a result, similar words share similar data representations, and we can leverage that similarity to explore the semantic space of a corpus, to encode documents with feature-rich data, and more."

In [4]:
def one_hot_encode(text):
    words = text.split()
    vocab = set(words)
    word_to_index = {word: idx for idx, word in enumerate(vocab)}
    one_hot_encoded = []
    for word in words:
        # print(f"Encoding word: '{word}'")
        one_hot_vector = [0] * len(vocab)
        one_hot_vector[word_to_index[word]] = 1
        print(f"One-hot vector for '{word}': {one_hot_vector}")
        one_hot_encoded.append(one_hot_vector)
    return one_hot_encoded, word_to_index, vocab

In [5]:
one_hot_encoded, word_to_index, vocabulary = one_hot_encode(example_text)
print("Vocabulary:", vocabulary)
print("Word to Index Mapping:", word_to_index)
print("One-Hot Encoded Matrix:")
for word, encoding in zip(example_text.split(), one_hot_encoded):
    print(f"{word}: {encoding}")

One-hot vector for 'Prior': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
One-hot vector for 'to': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
One-hot vector for 'the': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
One-hot vector for

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
documents = ["This is the first document.", "This document is the second document.",
              "And this is the third one.", "Is this the first document?"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

print("Bag-of-Words Matrix:")
print(X.toarray())
print("Vocabulary (Feature Names):", feature_names)

Bag-of-Words Matrix:
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]
Vocabulary (Feature Names): ['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

In [14]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embed_size)
        self.linear = nn.Linear(embed_size, vocab_size)

    def forward(self, context):
        context_embeds = self.embeddings(context).sum(dim=1)
        output = self.linear(context_embeds)
        return output

In [34]:
context_size = 2
raw_text = "word embeddings are awesome and more words here"
tokens = raw_text.split()
vocab = set(tokens)
word_to_index = {word: i for i, word in enumerate(vocab)}
print("word_to_index:", word_to_index)
data = []
if len(tokens) < 2*context_size + 1:
    print("Not enough tokens for the chosen context_size:", len(tokens))
else:
    for i in range(context_size, len(tokens) - context_size):
        context = [word_to_index[w] for w in tokens[i-context_size:i] + tokens[i+1:i+1+context_size]]
        print(f"Context words: {tokens[i-context_size:i] + tokens[i+1:i+1+context_size]}, Context indices: {context}")
        target = word_to_index[tokens[i]]
        print(f"Target word: '{tokens[i]}', Target index: {target}")
        data.append((torch.tensor(context), torch.tensor(target)))
print("Data samples (context indices, target index):")
for context, target in data:
    print(f"Context indices: {context.tolist()}, Target index: {target.item()}")

word_to_index: {'awesome': 0, 'embeddings': 1, 'and': 2, 'word': 3, 'more': 4, 'words': 5, 'are': 6, 'here': 7}
Context words: ['word', 'embeddings', 'awesome', 'and'], Context indices: [3, 1, 0, 2]
Target word: 'are', Target index: 6
Context words: ['embeddings', 'are', 'and', 'more'], Context indices: [1, 6, 2, 4]
Target word: 'awesome', Target index: 0
Context words: ['are', 'awesome', 'more', 'words'], Context indices: [6, 0, 4, 5]
Target word: 'and', Target index: 2
Context words: ['awesome', 'and', 'words', 'here'], Context indices: [0, 2, 5, 7]
Target word: 'more', Target index: 4
Data samples (context indices, target index):
Context indices: [3, 1, 0, 2], Target index: 6
Context indices: [1, 6, 2, 4], Target index: 0
Context indices: [6, 0, 4, 5], Target index: 2
Context indices: [0, 2, 5, 7], Target index: 4
